In [8]:
import numpy as np
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tqdm import tqdm

# Directories
CATALOG_DIR = "C:\\Users\\acer\\Documents\\study\\computer vision\\fashionRecommender\\fahion_dataset\\images"
USER_UPLOAD_DIR = "C:\\Users\\acer\\Documents\\study\\computer vision\\fashionRecommender\\fahion_dataset\\images\\10009"
IMAGE_SIZE = (224, 224)


# Load pretrained CNN model
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')
model = Model(inputs=base_model.input, outputs=base_model.output)

# Function to extract image feature vector
def extract_features(img_path):
    img = image.load_img(img_path, target_size=IMAGE_SIZE)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    features = model.predict(x, verbose=0)
    return features.flatten()

# Load all catalog images
catalog_paths = [os.path.join(CATALOG_DIR, f) for f in os.listdir(CATALOG_DIR) if f.endswith('.jpg')]
catalog_features = [extract_features(p) for p in tqdm(catalog_paths)]
catalog_features = np.array(catalog_features)

# User uploads 1 image
user_images = [os.path.join(USER_UPLOAD_DIR, f) for f in os.listdir(USER_UPLOAD_DIR) if f.endswith('.jpg')]

if len(user_images) == 0:
    print("Please upload an image to 'user_uploads/' folder.")
else:
    user_path = user_images[0]
    user_feature = extract_features(user_path).reshape(1, -1)

    # Compute similarity
    similarities = cosine_similarity(user_feature, catalog_features)[0]
    top_k = np.argsort(similarities)[::-1][:5]
    recommendations = [catalog_paths[i] for i in top_k]

    # Show uploaded image
    print("User Uploaded Image:")
    img = cv2.imread(user_path)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

    # Show Recommendations
    print("Recommended Similar Clothes:")
    plt.figure(figsize=(15, 5))
    for i, path in enumerate(recommendations):
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(1, 5, i+1)
        plt.imshow(img)
        plt.axis('off')
    plt.show()


ImportError: Traceback (most recent call last):
  File "C:\Users\acer\anaconda3\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.